# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FawadAhmad-bilal/flyrank-assignment-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Lane: **Refresh / Content Opportunity Scoring** — flagging low-performing content so an editor knows which pages to review first.

## 1. My lane as an ML task (type)

**Task type: Classification.**

The decision this improves: *which pages should an editor review first for refresh?* That is
answered by first sorting pages into "at risk / declining" vs "holding steady or growing," which
is a binary label problem, not a ranking-only or unsupervised problem. (I do rank the flagged
pages afterward using the model's predicted probability, but the core prediction — will this
page's demand keep dropping — is a yes/no classification.)

I'm not doing clustering (I'm not looking for undefined groups — I already know the two classes
I care about: declining vs not) and I'm not pure signal analysis (I need a page-level score an
editor can act on, not just a correlation writeup).

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 60)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Task type: Classification")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")

Task type: Classification
Rows: 30,000 | Columns: 44


## 2. Target or proxy

**Target: `is_declining_label`**, defined in the data dictionary as
`1 when trend_direction == "down"`, else `0`.

Where does it come from? `trend_direction` itself is a **rule**, not something the world directly
handed us — it's computed by comparing `impressions_last_30d` vs `impressions_prev_30d`
(down = more than a 20% drop). So strictly, this is a *defined proxy*, not a purely observed
outcome like "the client cancelled" or "traffic hit zero." I'm keeping it because it's the label
the whole starter dataset is built around and it's a reasonable stand-in for "demand is fading" —
but I note the caveat: a page can cross the -20% threshold from ordinary noise, not real decline.
I will **not** use `trend_direction` or `trend_pct` as a feature — that would leak the label into
the inputs, since the label is defined directly from those columns.

In [2]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print()
print(f"Base rate (share declining): {df['is_declining_label'].mean():.1%}")

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Base rate (share declining): 54.2%


## 3. Success metric

**Primary metric: precision@K** (top ~20% of pages ranked by predicted decline probability),
because an editor only has time to review a limited queue each week — what matters is whether
the pages at the *top* of that queue are actually declining, not how the model does on average
across all 30,000 pages.

**Secondary/defend-the-model metric: ROC-AUC**, to check the model separates the two classes at
all before I bother building a queue out of it.

I'm avoiding plain "accuracy" — with the classes roughly balanced (54% vs 46%) it wouldn't be
misleading here, but it also wouldn't tell an editor anything about queue quality, which is the
actual decision this task supports. Both metrics need a floor to beat: a random/no-model queue,
computed below.

In [3]:
base_rate = df["is_declining_label"].mean()
k = int(len(df) * 0.20)  # a realistic weekly review queue: top 20% of pages

# The floor both metrics must clear: a random/no-model queue would score the base rate
# on precision@K, and 0.5 on ROC-AUC (a coin flip can't separate the classes at all).
print(f"Rows: {len(df):,} | Review queue size (top 20%): {k:,} pages")
print(f"Base rate (no-model precision@{k} floor): {base_rate:.1%}")
print("Random-guess ROC-AUC floor: 0.500")
print("-> a real model only earns its place if it clears both floors by a real margin.")

Rows: 30,000 | Review queue size (top 20%): 6,000 pages
Base rate (no-model precision@6000 floor): 54.2%
Random-guess ROC-AUC floor: 0.500
-> a real model only earns its place if it clears both floors by a real margin.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (page)**, for one client, aggregated over a trailing 90-day window.
Below is the actual slice I'll work from: identifiers, the safe (non-leaking) features I'd feed a
model, and the label — with `trend_direction` / `trend_pct` kept alongside only for reference,
clearly marked as label-source, not features.

In [4]:
unit_of_analysis_cols = [
    "content_id", "client_id",                         # identifiers - grouping only
    "content_type", "main_intent", "word_count",        # content properties
    "impressions_90d", "clicks_90d", "ctr", "avg_position",   # 90d activity/quality
    "engagement_rate", "ai_traffic_pct",                # engagement
    "content_age_days", "days_since_last_update",       # freshness
    "trend_direction", "trend_pct",                     # LABEL SOURCE - never a feature
    "is_declining_label",                                # the target
]

sample = df[unit_of_analysis_cols].head(5)
sample

             content_id          client_id     content_type    main_intent  \
0  content_304f48230142  client_f369cb89fc  keyword article  transactional  
1  content_a1fb4e703a9e  client_4e07408562  keyword article  informational  
2  content_9aa793d4d895  client_7f2253d7e2  keyword article  informational  
3  content_331d6c4de07b  client_19581e27de  keyword article     commercial  
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article  informational  

   word_count  impressions_90d  clicks_90d   ctr  avg_position  \
0      3221.0             3803          29  0.76          10.6  
1      2481.0            15320           7  0.05          20.3  
2      3515.0            12581          11  0.09          36.5  
3         NaN            11751          58  0.49           6.2  
4      2803.0            19140          24  0.13          44.0  

   engagement_rate  ai_traffic_pct  content_age_days  days_since_last_update  \
0             5.88             0.0               187            

## 5. Why ML beats a fixed rule here

A fixed rule ("flag anything with `avg_position` worse than 20 and `word_count` under 1,000")
sounds reasonable but breaks in practice, because the signals that predict decline **interact and
shift by content type and client**:

- A thin, low-position page can still be perfectly stable if it gets almost no search demand to
  begin with (nothing to lose).
- A long, well-ranked page can decline anyway if AI-answer traffic (`ai_traffic_pct`) is quietly
  eating its clicks — a pattern a one-line rule won't catch because it needs several signals
  weighed together, not one threshold.
- The relationship between `engagement_rate`, `content_age_days`, and decline isn't the same
  across all 32 clients or all 3 content types — a single global if-statement either overfits to
  the majority pattern or misses smaller segments entirely.

That's exactly the "many signals, tangled, shifting" case where a model that can weigh and combine
features (logistic regression or gradient boosting) earns its keep over a hand-written rule — while
still staying explainable enough to hand an editor reason codes, not just a black-box score.

In [5]:
# Quick evidence: a single-threshold rule (word_count < 1000) barely moves off the base rate,
# because it ignores every other signal.
rule_flag = (df["word_count"].fillna(0) < 1000).astype(int)
rule_precision = df.loc[rule_flag == 1, "is_declining_label"].mean()
rule_recall = df.loc[rule_flag == 1, "is_declining_label"].sum() / df["is_declining_label"].sum()

print(f"Rule 'word_count < 1000' flags {rule_flag.sum():,} pages")
print(f"  precision: {rule_precision:.1%}  (base rate: {df['is_declining_label'].mean():.1%})")
print(f"  recall:    {rule_recall:.1%}")
print("-> one signal alone doesn't separate the classes; it needs to be combined with others.")

Rule 'word_count < 1000' flags 8,672 pages
  precision: 43.6%  (base rate: 54.2%)
  recall:    23.3%
-> one signal alone doesn't separate the classes; it needs to be combined with others.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.